## INITIALIZATION

After building 0813 notebook (demographic + medication + comorbidities + cost derivatives), this notebook creates treatment adherence features appropriate for Stage 3-4 CKD.

In [1]:
import sys
print(f"Python version: {sys.version}")
import json
import logging
import csv
import gzip
import re
import pandas as pd
import numpy as np
from functools import reduce
from pyspark.sql.types import StringType,DecimalType,DoubleType,IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.feature import VectorAssembler, Bucketizer
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql import SparkSession,Row
from pyspark import SparkConf
import plotly.express as px
import plotly.graph_objects as go
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# -----------------------------------------------------------------------------
# INITIALIZE LOGGING
# -----------------------------------------------------------------------------
f = '%(asctime)-15s %(levelname)-8s %(message)s'
logger = logging.getLogger(__name__)
logger.setLevel("DEBUG")
logging.basicConfig(format=f)


from IPython.core.magic import register_cell_magic

# -----------------------------------------------------------------------------
# start_spark
# -----------------------------------------------------------------------------
def start_spark(
    driver_memory="100g",
    storage_fraction=0.5,
    num_nodes=10,
):
    """Initialize spark

    Arguments:
        driver_memory: Maximum heap size for the Spark driver Java
            virtual machine.
        storage_fraction: Controls what portion of Spark's unified
            memory is reserved for storage (i.e., caching/persisting data
            and broadcast variables), as a fraction of the total
            execution + storage memory pool.
            If you cache/persist a lot of data, and you're evicting
            data too early, you might increase this value (e.g. 0.6 or 0.7).
            Conversely, if your job is shuffle-heavy and fails due to
            memory pressure, you might decrease it (e.g. 0.3).
        num_nodes: How many concurrent threads to use while running
            in "local mode" (i.e. in a single machine instead of a cluster).
            Use '*' to use all cores, or an integer > 0 for a specific
            number of threads.
    """

    conf = SparkConf().setAppName("My_Application")
    conf.set("spark.driver.memory", driver_memory)
    conf.set("spark.memory.storageFraction", str(storage_fraction))
    conf.setMaster(f"local[{num_nodes}]")

    spark = SparkSession.builder.config(conf=conf).getOrCreate()
    spark.sparkContext.setLogLevel('WARN')

    return spark


spark = start_spark(num_nodes=10)
#spark.stop()

  
@register_cell_magic
def spark_sql(line, cell):
    result = spark.sql(cell)
    result.show(n=1000)
  

# -- READ ENROLLMENT AND DATA TABLES
enrollment_file = f"/Users/Charles/DATA/ckd/ckd_enrollment"
logger.info(f">>> Reading enrollment file: {enrollment_file}")
df_enrollment = spark.read.format("parquet").load(enrollment_file)
df_enrollment.createOrReplaceTempView('enrollment')
logger.info(f">>> ENROLLMENT has {df_enrollment.count():,} rows")
logger.info(f">>> ENROLLMENT has {df_enrollment.select('ENROLID').distinct().count():,} unique enrollees")

claims_file = f"/Users/Charles/DATA/ckd/ckd_claims"
logger.info(f">>> Reading claims file: {claims_file}")
df_claims = spark.read.format("parquet").load(claims_file)
df_claims.createOrReplaceTempView('claims')
logger.info(f">>> CLAIMS has {df_claims.count():,} rows")
logger.info(f">>> CLAIMS has {df_claims.select('ENROLID').distinct().count():,} unique enrollees")

# -- NOTE: with the "createOrReplaceTempView" we define a view of these
# -- tables, so we can use them in SQL queries.


Python version: 3.11.0 (main, Jun 13 2025, 14:48:45) [Clang 16.0.0 (clang-1600.0.26.6)]


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/04 17:11:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/04 17:11:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/04 17:11:10 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
2025-11-04 17:11:11,599 INFO     >>> Reading enrollment file: /Users/Charles/DATA/ckd/ckd_enrollment
25/11/04 17:11:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
2025-11-04 17:11:13,250 INFO     >>> ENROLLMENT has 2,586,930 rows
2025-11-04 17:11:15,002 INFO     >>> ENROLLMENT has 862,310 unique enrollees    
2025-11-04

# LOAD BASE DATA 

In [ ]:
#pilot_ids=spark.read.format("parquet").load( "0831_2017_18_highcost_cutoffs.parquet")
pilot_ids = spark.read.format("parquet").load( "df_1102_w_cost_derivatives_stage56.parquet")

pilot_claims = (
    df_claims
      .join(pilot_ids, on="ENROLID", how="inner")
      .orderBy("ENROLID","SVCDATE")
)


In [3]:

def ckd_stage_expr(dx_cols):
    """
    Returns a single Spark expression that gives the CKD stage as:
      1-6  … exact stage found
      0    … N189 (unspecified CKD)
      null … no CKD code in any DX column
    
    This function uses regex to handle CKD subcodes automatically
    and prioritizes diagnosis columns from left to right.
    """
    pieces = []
    for c in dx_cols:
        pieces.append(
            F.when(F.col(c) == "N189", F.lit(0))
            .otherwise(
                F.expr(f"try_cast(regexp_extract({c}, 'N18([1-6]).*', 1) AS int)")
            )
        )
    # coalesce from left to right → first non-null wins
    return F.coalesce(*pieces)


In [4]:
def create_stage34_monitoring_adherence(pilot_claims, dx_cols, year=2017):
    """
    Create adherence features for Stage 3-4 CKD monitoring guidelines
    
    Guidelines recommend:
    - Stage 3: Lab monitoring every 3-6 months
    - Stage 4: Lab monitoring every 3 months
    - Annual nephrology consultation
    
    Args:
        pilot_claims: Claims data filtered to pilot cohort
        dx_cols: List of diagnosis columns
        year: Year to analyze
        
    Returns:
        DataFrame with monitoring adherence features
    """
    
    print(f"=== CREATING STAGE 3-4 CKD MONITORING ADHERENCE FOR {year} ===")
    
    # Identify Stage 3-4 patients and their first diagnosis date
    stage34_patients = (
        pilot_claims
        .filter(F.col("YEAR") == year)
        .withColumn("ckd_stage", ckd_stage_expr(dx_cols))
        .filter(F.col("ckd_stage").isin([3, 4]))
        .groupBy("ENROLID")
        .agg(
            F.min("SVCDATE").alias("first_ckd_date"),
            F.count("*").alias("ckd_visit_count")
        )
    )
    
    # -----------------------------------------------------------------------------
    # 1. LAB MONITORING ADHERENCE
    # -----------------------------------------------------------------------------
    
    # Key lab tests CPT codes for CKD monitoring
    ckd_lab_codes = [
        # Kidney function tests
        "82565",  # Creatinine blood
        "82575",  # Creatinine clearance
        "84520",  # Urea nitrogen (BUN)
        "81000", "81001", "81002", "81003",  # Urinalysis
        "82570",  # Creatinine urine
        "84156",  # Protein urine
        "84166",  # Protein electrophoresis
        
        # GFR estimation
        "82565",  # Used for eGFR calculation
        
        # Electrolytes and minerals
        "84132",  # Potassium
        "84295",  # Sodium
        "82310",  # Calcium
        "84100",  # Phosphorus
        "83735",  # Magnesium
        
        # Anemia monitoring
        "85025", "85027",  # Complete blood count
        "83550",  # Iron studies
        "82728",  # Ferritin
        "84466",  # Transferrin
        
        # Bone metabolism
        "83970",  # Parathyroid hormone (PTH)
        "82306",  # Vitamin D
        
        # Metabolic panel
        "80047", "80048", "80053"  # Comprehensive metabolic panel
    ]
    
    # Count lab tests per quarter
    lab_monitoring = (
        pilot_claims
        .filter(F.col("YEAR") == year)
        .filter(F.col("PROC1").isin(ckd_lab_codes))
        .withColumn("quarter", F.quarter("SVCDATE"))
        .groupBy("ENROLID", "quarter")
        .count()
        .groupBy("ENROLID")
        .agg(
            F.count("quarter").alias("quarters_with_labs"),
            F.sum("count").alias("total_lab_tests")
        )
    )
    
    # Join with stage info and calculate adherence
    lab_adherence = (stage34_patients
        .join(lab_monitoring, on="ENROLID", how="left")
        .fillna({"quarters_with_labs": 0, "total_lab_tests": 0})
        .withColumn(
        "lab_monitoring_adherent",
        F.when(F.col("quarters_with_labs") >= 2, 1).otherwise(0)
        )
        .withColumn(
            "lab_monitoring_intensity",
            F.when(F.col("total_lab_tests") >= 12, "High")
            .when(F.col("total_lab_tests") >= 6, "Moderate")
            .when(F.col("total_lab_tests") >= 3, "Low")
            .otherwise("Minimal")
        ))
    
    # -----------------------------------------------------------------------------
    # 2. NEPHROLOGY CONSULTATION ADHERENCE
    # -----------------------------------------------------------------------------
    
    # Nephrology visit CPT codes
    nephrology_codes = [
        # Office visits that could be nephrology (need to combine with provider specialty if available)
        "99201", "99202", "99203", "99204", "99205",  # New patient
        "99211", "99212", "99213", "99214", "99215",  # Established patient
        
        # Consultation codes
        "99241", "99242", "99243", "99244", "99245",  # Office consultation
        
        # Kidney-specific procedures often done by nephrologists
        "90960", "90961", "90962",  # ESRD-related monthly services (pre-dialysis counseling)
        "90966", "90967", "90968", "90969", "90970"  # CKD education services
    ]
    
    # Count nephrology consultations
    nephrology_visits = (
        pilot_claims
        .filter(F.col("YEAR") == year)
        .filter(
            F.col("PROC1").isin(nephrology_codes) |
            # Also check if STDPROV indicates nephrology (code 39 typically)
            (F.col("STDPROV") == 39)
        )
        .groupBy("ENROLID")
        .agg(
            F.count("*").alias("nephrology_visit_count"),
            F.min("SVCDATE").alias("first_nephrology_date")
        )
    )
    
    # Calculate nephrology adherence
    nephrology_adherence = (
        lab_adherence
        .join(nephrology_visits, on="ENROLID", how="left")
        .fillna({"nephrology_visit_count": 0})
        .withColumn(
            "nephrology_consult_adherent",
            F.when(F.col("nephrology_visit_count") >= 1, 1).otherwise(0)
        )
        .withColumn(
            "days_to_nephrology",
            F.datediff(F.col("first_nephrology_date"), F.col("first_ckd_date"))
        )
        .withColumn(
            "early_nephrology_referral",
            F.when(
                (F.col("days_to_nephrology") >= 0) & 
                (F.col("days_to_nephrology") <= 90), 1
            ).otherwise(0)
        )
    )
    
    return nephrology_adherence


dx_cols = ["DX1", "DX2", "DX3", "DX4", "PDX"]  # Your diagnosis columns
year = 2017
adherence_features = create_stage34_monitoring_adherence(pilot_claims, dx_cols, year)


=== CREATING STAGE 3-4 CKD MONITORING ADHERENCE FOR 2017 ===


In [16]:
adherence_features.count()

33585

In [6]:
comprehensive_adherence = (
        pilot_ids
        .join(adherence_features, on="ENROLID", how="left")
        .fillna(0)
    ) # Summary statistics
print("\n=== ADHERENCE SUMMARY FOR STAGE 3-4 CKD ===")
comprehensive_adherence.columns



=== ADHERENCE SUMMARY FOR STAGE 3-4 CKD ===


['ENROLID',
 '2017Q1_ckd_cost',
 '2017Q1_ckd_claims',
 '2017Q1_max_ckd_stage',
 '2017Q1_direct_ckd_cost',
 '2017Q1_procedure_ckd_cost',
 '2017Q1_comorbidity_ckd_cost',
 '2017Q2_ckd_cost',
 '2017Q2_ckd_claims',
 '2017Q2_max_ckd_stage',
 '2017Q2_direct_ckd_cost',
 '2017Q2_procedure_ckd_cost',
 '2017Q2_comorbidity_ckd_cost',
 '2017Q3_ckd_cost',
 '2017Q3_ckd_claims',
 '2017Q3_max_ckd_stage',
 '2017Q3_direct_ckd_cost',
 '2017Q3_procedure_ckd_cost',
 '2017Q3_comorbidity_ckd_cost',
 '2017Q4_ckd_cost',
 '2017Q4_ckd_claims',
 '2017Q4_max_ckd_stage',
 '2017Q4_direct_ckd_cost',
 '2017Q4_procedure_ckd_cost',
 '2017Q4_comorbidity_ckd_cost',
 'total_ckd_cost_2017',
 'ckd_cost_trend_2017',
 'ckd_cost_volatility_2017',
 'ckd_cost_deriv_Q1_Q2_2017',
 'ckd_cost_deriv_Q2_Q3_2017',
 'ckd_cost_deriv_Q3_Q4_2017',
 'is_increasing_Q1_Q2_2017',
 'is_increasing_Q2_Q3_2017',
 'is_increasing_Q3_Q4_2017',
 'total_increasing_quarters_2017',
 'is_consistently_increasing_2017',
 'is_consistently_decreasing_2017',
 'a

In [7]:
adherence_features.groupBy("lab_monitoring_intensity").agg(
    F.count("*").alias("patient_count"),
    F.mean("total_lab_tests").alias("avg_lab_tests"),
).show()

+------------------------+-------------+------------------+
|lab_monitoring_intensity|patient_count|     avg_lab_tests|
+------------------------+-------------+------------------+
|                    High|         2388|  51.8856783919598|
|                     Low|          132| 3.977272727272727|
|                 Minimal|          190|0.6105263157894737|
|                Moderate|          319| 8.589341692789969|
+------------------------+-------------+------------------+



In [8]:
enrol_info = (
    df_enrollment
      .select("ENROLID","MEDIAN_INCOME","INCOME_LEVEL")
      .join(comprehensive_adherence, on="ENROLID", how="inner")
      .dropDuplicates(["ENROLID"])
      .drop("INDSTRY","MEDIAN_INCOME","quarters_with_labs",'ckd_visit_count', 'days_to_nephrology')
      .filter(col("annual_cost17").isNotNull() & (col("annual_cost17") >= 0))
)
enrol_info = enrol_info.fillna({"INCOME_LEVEL": 0})
enrol_pd = enrol_info.toPandas()
#enrol_info.write.mode("overwrite").parquet("0902_adherence_income_info.parquet")

enrol_info.write.mode("overwrite").parquet("df_1102_w_adherence_features_stage56.parquet")


### Optional: Medication Management Adherence + Comorbidity Management
ACE/ARB adherence (critical for slowing progression)

Blood pressure control medications

Statins for cardiovascular protection

Phosphate binders and vitamin D (for Stage 4)


Diabetes monitoring (HbA1c quarterly if diabetic)

Hypertension management

Anemia monitoring and treatment

In [ ]:

def create_medication_management_adherence(pilot_claims, dx_cols, year=2017):
    """
    Create medication adherence features for Stage 3-4 CKD management
    Focus on guideline-recommended medications for progression prevention
    
    Args:
        pilot_claims: Claims data with RX records
        dx_cols: List of diagnosis columns
        year: Year to analyze
        
    Returns:
        DataFrame with medication management adherence features
    """
    
    print(f"=== CREATING STAGE 3-4 MEDICATION ADHERENCE FOR {year} ===")
    
    # Get Stage 3-4 patients
    stage34_patients = (
        pilot_claims
        .filter(F.col("YEAR") == year)
        .select("ENROLID")
        .distinct()
    )
    
    # -----------------------------------------------------------------------------
    # KEY MEDICATIONS FOR CKD PROGRESSION PREVENTION
    # -----------------------------------------------------------------------------
    
    # Based on KDIGO guidelines for Stage 3-4 CKD
    ckd_medications = {
        "ace_arb": {
            "thercls": ["172", "173"],  # ACE inhibitors, ARBs
            "importance": "critical",  # Renoprotective, slows progression
            "target_pdc": 0.80  # 80% adherence threshold
        },
        "bp_control": {
            "thercls": ["174", "175", "176", "177"],  # Diuretics, beta blockers, CCBs
            "importance": "high",
            "target_pdc": 0.80
        },
        "statin": {
            "thercls": ["050"],  # Statins for cardiovascular protection
            "importance": "high",
            "target_pdc": 0.80
        },
        "vitamin_d": {
            "thercls": ["241"],  # Active vitamin D
            "importance": "moderate", 
            "target_pdc": 0.70
        },
        "anemia_treatment": {
            "thercls": ["082"],  # EPO/ESA for anemia
            "importance": "conditional",  # Only if anemic
            "target_pdc": 0.80
        }
    }
    
    # Get prescription claims
    rx_claims = (
        pilot_claims
        .filter(F.col("claim_type") == "RX")
        .filter(F.col("YEAR") == year)
    )
    
    # Calculate medication adherence metrics
    adherence_dfs = []
    
    for med_class, config in ckd_medications.items():
        med_data = (
            stage34_patients
            .join(
                rx_claims
                .filter(F.col("THERCLS").isin(config["thercls"]))
                .groupBy("ENROLID")
                .agg(
                    F.count("*").alias(f"{med_class}_fills"),
                    F.sum("DAYSUPP").alias(f"{med_class}_days_supply"),
                    F.countDistinct(F.month("SVCDATE")).alias(f"{med_class}_months_filled")
                ),
                on="ENROLID",
                how="left"
            )
            .fillna({
                f"{med_class}_fills": 0,
                f"{med_class}_days_supply": 0,
                f"{med_class}_months_filled": 0
            })
            .withColumn(
                f"{med_class}_pdc",
                F.least(F.col(f"{med_class}_days_supply") / 365, F.lit(1.0))
            )
            .withColumn(
                f"{med_class}_adherent",
                F.when(
                    F.col(f"{med_class}_pdc") >= config["target_pdc"], 1
                ).otherwise(0)
            )
            .select(
                "ENROLID",
                f"{med_class}_fills",
                f"{med_class}_pdc",
                f"{med_class}_adherent"
            )
        )
        adherence_dfs.append(med_data)
    
    # Combine all medication features
    medication_features = reduce(
        lambda df1, df2: df1.join(df2, on="ENROLID", how="outer"),
        adherence_dfs
    )
    
    # Create composite medication adherence score
    medication_features = medication_features.withColumn(
        "critical_med_adherent",
        F.col("ace_arb_adherent")  # ACE/ARB is most critical
    ).withColumn(
        "medication_adherence_score",
        (
            F.col("ace_arb_pdc") * 0.3 +  # Weight critical medications higher
            F.col("bp_control_pdc") * 0.25 +
            F.col("statin_pdc") * 0.20 +
            F.col("phosphate_binder_pdc") * 0.10 +
            F.col("vitamin_d_pdc") * 0.10 +
            F.col("anemia_treatment_pdc") * 0.05
        )
    )
    
    return medication_features


def create_comorbidity_management_adherence(pilot_claims, dx_cols, year=2017):
    """
    Track adherence to managing key CKD comorbidities
    Important for Stage 3-4 to prevent progression
    
    Args:
        pilot_claims: Claims data
        dx_cols: Diagnosis columns
        year: Year to analyze
        
    Returns:
        DataFrame with comorbidity management features
    """
    
    print(f"=== CREATING COMORBIDITY MANAGEMENT ADHERENCE FOR {year} ===")
    
    # Get Stage 3-4 patients with comorbidities
    stage34_with_comorbidities = (
        pilot_claims
        .filter(F.col("YEAR") == year)
    )
    
    # -----------------------------------------------------------------------------
    # KEY COMORBIDITIES REQUIRING MANAGEMENT
    # -----------------------------------------------------------------------------
    
    # 1. DIABETES MANAGEMENT (if diabetic)
    diabetes_codes = ["E119", "E109", "E1122", "E1022"]  # From your ckd_comorbidity_codes
    
    diabetes_management = (
        stage34_with_comorbidities
        .filter(reduce(lambda a, b: a | b, 
                      [F.col(dx) == code for dx in dx_cols for code in diabetes_codes]))
        .select("ENROLID")
        .distinct()
        .join(
            pilot_claims
            .filter(F.col("YEAR") == year)
            .filter(
                # HbA1c monitoring (should be quarterly)
                F.col("PROC1").isin(["83036", "83037"]) |
                # Glucose monitoring
                F.col("PROC1").isin(["82947", "82948", "82950"])
            )
            .groupBy("ENROLID")
            .agg(
                F.count("*").alias("diabetes_monitoring_count"),
                F.countDistinct(F.quarter("SVCDATE")).alias("quarters_with_diabetes_monitoring")
            ),
            on="ENROLID",
            how="left"
        )
        .fillna(0)
        .withColumn(
            "diabetes_monitoring_adherent",
            F.when(F.col("quarters_with_diabetes_monitoring") >= 3, 1).otherwise(0)
        )
    )
    
  
    # 3. ANEMIA MANAGEMENT (common in CKD)
    anemia_codes = ["D631", "D6310", "D6311", "D6312", "D6313", "D6314", "D509", "D649"]
    
    anemia_management = (
        stage34_with_comorbidities
        .filter(reduce(lambda a, b: a | b,
                      [F.col(dx) == code for dx in dx_cols for code in anemia_codes]))
        .select("ENROLID")
        .distinct()
        .join(
            pilot_claims
            .filter(F.col("YEAR") == year)
            .filter(
                # Iron studies, CBC
                F.col("PROC1").isin(["83550", "82728", "84466", "85025", "85027"])
            )
            .groupBy("ENROLID")
            .agg(F.count("*").alias("anemia_monitoring_count")),
            on="ENROLID",
            how="left"
        )
        .fillna(0)
        .withColumn(
            "anemia_monitoring_adherent",
            F.when(F.col("anemia_monitoring_count") >= 2, 1).otherwise(0)
        )
    )
    
    # Combine comorbidity management features
    all_patients = stage34_with_comorbidities.select("ENROLID").distinct()
    
    comorbidity_features = (
        all_patients
        .join(diabetes_management, on="ENROLID", how="left")
        .join(anemia_management, on="ENROLID", how="left")
        .fillna(0)
    )
    
    return comorbidity_features


def create_comprehensive_stage34_adherence(pilot_claims, dx_cols, year=2017):
    """
    Combine all Stage 3-4 CKD adherence measures into comprehensive features
    
    Returns features indicating whether patients are following recommended
    care pathways for CKD progression prevention
    """
    
    print(f"=== CREATING COMPREHENSIVE STAGE 3-4 ADHERENCE FEATURES FOR {year} ===")
    
    # Get all component adherence measures
    monitoring_adherence = create_stage34_monitoring_adherence(pilot_claims, dx_cols, year)
    #medication_adherence = create_medication_management_adherence(pilot_claims, dx_cols, year)
    #comorbidity_adherence = create_comorbidity_management_adherence(pilot_claims, dx_cols, year)
    
    # Get all patients for left join
    all_patients = pilot_claims.select("ENROLID").distinct()
    
    # Combine all features
    comprehensive_adherence = (
        all_patients
        .join(monitoring_adherence, on="ENROLID", how="left")
        #.join(medication_adherence, on="ENROLID", how="left")
        #.join(comorbidity_adherence, on="ENROLID", how="left")
        .fillna(0)
    )
    
    # Create composite adherence indicators
    comprehensive_adherence = comprehensive_adherence.withColumn(
        "overall_adherence_score",
        (
            F.col("lab_monitoring_adherent") * 0.25 +
            F.col("nephrology_consult_adherent") * 0.20 +
            F.col("critical_med_adherent") * 0.30 +
            F.col("diabetes_monitoring_adherent") * 0.15 +
            F.col("anemia_monitoring_adherent") * 0.10
        )
    ).withColumn(
        "adherence_level",
        F.when(F.col("overall_adherence_score") >= 0.75, "High")
        .when(F.col("overall_adherence_score") >= 0.50, "Moderate")
        .when(F.col("overall_adherence_score") >= 0.25, "Low")
        .otherwise("Minimal")
    ).withColumn(
        "guideline_adherent",
        F.when(
            (F.col("lab_monitoring_adherent") == 1) &
            (F.col("critical_med_adherent") == 1), 1
        ).otherwise(0)
    )
    
    # Add risk flags
    comprehensive_adherence = comprehensive_adherence.withColumn(
        "high_risk_nonadherent",
        F.when(
            (F.col("stage_2017") == 4) &
            (F.col("lab_monitoring_adherent") == 0) &
            (F.col("nephrology_consult_adherent") == 0), 1
        ).otherwise(0)
    ).withColumn(
        "progression_risk_score",
        F.when(F.col("stage_2017") == 4, 2).otherwise(1) *
        (2 - F.col("overall_adherence_score"))  # Higher score = higher risk
    )
    
    # Summary statistics
    print("\n=== ADHERENCE SUMMARY FOR STAGE 3-4 CKD ===")
    comprehensive_adherence.groupBy("adherence_level").agg(
        F.count("*").alias("patient_count"),
        F.mean("total_lab_tests").alias("avg_lab_tests"),
        F.mean("medication_adherence_score").alias("avg_med_score")
    ).show()
    
    return comprehensive_adherence
